# P03-P10 · Conciliar Metro y medir correspondencias con buses

> **¿Qué estaciones de Metro tienen buses a 200 m y cuánto depende la respuesta de la fuente?**

GTFS Red Movilidad y el archivo público de estaciones de Metro para carga bip!.

Este cuaderno corre de principio a fin en Colab. La explicación completa, con las hojas de Excel dibujadas, está en [CUADERNO.md](https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab/blob/main/cases/correspondencia_metro/CUADERNO.md).

*Generado por `scripts/build_cuadernos.py`. No editar a mano: los cambios se pierden al regenerar.*


In [ ]:
# Preparación: clona el repositorio y descarga los datos públicos.
# Los datos NO vienen del repositorio: se descargan del portal de datos
# abiertos, porque el laboratorio no redistribuye datos de terceros.
!git clone --depth 1 https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab.git lab-dtpm
%cd lab-dtpm
!pip install --quiet 'pandas>=2.2' 'numpy>=1.26' openpyxl XlsxWriter matplotlib
!python scripts/fetch_data.py gtfs-red-movilidad bip-lugares-carga


## 1 · El problema

Dos fuentes públicas describen las estaciones de Metro con granos diferentes: el GTFS
tiene 126 estaciones y el archivo bip! tiene 143 filas, una por estación y línea. La
diferencia no es un error. El problema empieza cuando se cruzan como si cada fila
representara lo mismo.

La conciliación usa dos criterios independientes —nombre normalizado y proximidad— y
convierte sus desacuerdos en casos para revisar. Esto descubre once estaciones del
archivo bip! a más de 500 metros de la ubicación GTFS y una separación de 26,2 km entre
dos filas que dicen «Plaza de Armas». Luego se responde la pregunta operativa: qué
estaciones tienen buses a 200 metros y cuántos servicios distintos las atienden.


## 2 · La solución que se va a aplicar

Se diagnostica el grano antes del cruce, se reducen las filas bip! a estaciones sin
ocultar su dispersión y se confronta nombre con distancia. Cuatro detectores de calidad
declaran qué comparan y cuál es su punto ciego. Solo después se calculan las paradas y
servicios de bus cercanos. La respuesta se repite con la otra fuente de coordenadas
para medir cuánto depende del proveedor elegido.


## Implementaciones comparables

La ruta principal usa Pandas. Polars y R recalculan el indicador central desde el contrato de salida versionado y generan el mismo gráfico.


### Polars + Matplotlib

```python
"""Contraste Polars y visualización Matplotlib del caso correspondencia_metro."""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

ROOT = Path(__file__).resolve().parents[3] if "__file__" in globals() else Path.cwd()
SOURCE = ROOT / "cases" / "correspondencia_metro" / "resultados" / "detectores.csv"


def analizar() -> pl.DataFrame:
    datos = pl.read_csv(SOURCE)
    return datos.sort("hallazgos", descending=True).select(["detector", "hallazgos"])


def graficar(resultado: pl.DataFrame, destino: Path) -> Path:
    x = resultado["detector"].to_list()
    y = resultado["hallazgos"].to_list()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh([str(value) for value in x], y, color="#11253B")
    ax.set_xlabel("Hallazgos (recuento)")
    ax.set_ylabel("Detector")
    ax.set_xlim(left=0)
    ax.set_title("Cada detector encuentra un punto ciego distinto")
    ax.grid(axis="y", alpha=0.25)
    fig.text(
        0.01,
        0.01,
        "Fuente: Resultados reproducibles del caso P03/P10 · Período: corte publicado",
        fontsize=8,
        color="#58677A",
    )
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    fig.savefig(destino, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return destino


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--grafico", type=Path)
    args = parser.parse_args()
    resultado = analizar()
    print(resultado)
    if args.grafico:
        print(graficar(resultado, args.grafico))


if __name__ == "__main__":
    main()
```

Ejecutar con `uv run --extra polars python cases/correspondencia_metro/cuaderno/solucion_polars.py`.


### R + ggplot2

```r
#!/usr/bin/env Rscript
# Contraste R + ggplot2 del caso correspondencia_metro.
script_arg <- grep("^--file=", commandArgs(trailingOnly = FALSE), value = TRUE)
script_path <- if (length(script_arg)) sub("^--file=", "", script_arg[1]) else "cuaderno/solucion.R"
root <- normalizePath(file.path(dirname(script_path), "../../.."), mustWork = TRUE)
source_csv <- file.path(root, "cases", "correspondencia_metro", "resultados", "detectores.csv")

datos <- read.csv(source_csv, check.names = FALSE, fileEncoding = "UTF-8")
resultado <- datos[order(-datos$hallazgos), c("detector", "hallazgos")]
print(resultado, row.names = FALSE)

args <- commandArgs(trailingOnly = TRUE)
plot_pos <- match("--grafico", args)
if (!is.na(plot_pos)) {
  if (plot_pos == length(args)) stop("--grafico necesita una ruta de salida")
  if (!requireNamespace("ggplot2", quietly = TRUE)) {
    stop("Instale ggplot2 para generar el gráfico: install.packages('ggplot2')")
  }
  destino <- args[plot_pos + 1]
  grafico <- ggplot2::ggplot(
    resultado,
    ggplot2::aes(x = .data[["detector"]], y = .data[["hallazgos"]])
  ) +
    ggplot2::geom_col(fill = "#11253B") +
    ggplot2::coord_flip() +
    ggplot2::labs(
      title = "Cada detector encuentra un punto ciego distinto",
      x = "Hallazgos (recuento)", y = "Detector",
      caption = "Fuente: Resultados reproducibles del caso P03/P10 · Período: corte publicado"
    ) +
    ggplot2::theme_minimal(base_size = 11)
  ggplot2::ggsave(destino, grafico, width = 9, height = 5, dpi = 150)
  cat(destino, "\n")
}
```

Ejecutar con `Rscript cases/correspondencia_metro/cuaderno/solucion.R`.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cases.correspondencia_metro import solution as p10

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 30)


## 3 · Los pasos


### Paso 1 · Diagnosticar el grano de las dos fuentes


126 y 143 se reconcilian explicando qué representa cada fila.

> ⚠️ Forzar una unión uno a uno elimina 17 filas legítimas de estaciones de combinación.


In [ ]:
resultados = p10.analizar()
resultados['grano']


### Paso 2 · Conciliar por nombre y por distancia


Dos criterios independientes permiten medir cobertura y desacuerdo.

> ⚠️ El nombre encuentra 125 de 126; la fila restante no debe desaparecer en silencio.


In [ ]:
resultados['conciliacion']


### Paso 3 · Revisar donde los criterios discrepan


Los trece desacuerdos son el conjunto pequeño que requiere inspección.

> ⚠️ Elegir automáticamente el criterio con más coincidencias puede confirmar una coordenada corrupta.


In [ ]:
resultados['criterios_discrepantes']


### Paso 4 · Declarar el punto ciego de cada detector


Calidad significa saber qué error puede y qué error no puede observar cada control.

> ⚠️ UTM y lat/lon pueden estar ambas equivocadas de la misma forma: el detector interno no lo ve.


In [ ]:
resultados['detectores']


### Paso 5 · Medir la correspondencia a 200 metros


Cada estación se cruza con paradas de bus por distancia haversine.

**En Excel:** `=CONTAR.SI.CONJUNTO(estacion;A2;distancia_m;"<=200")`

Debe dar: **126 de 126 estaciones con al menos una parada**

**En Google Sheets:** la misma fórmula, sin cambios.

> ⚠️ Contar filas de horarios multiplica un mismo servicio; la unidad es servicio distinto.


In [ ]:
resultados['correspondencia']


### Paso 6 · Mostrar la distribución de servicios


La mediana, el p90 y el máximo evitan resumir toda la red con un único extremo.

> ⚠️ El máximo de 55 describe Santa Lucía, no la estación típica; la mediana es 13.


In [ ]:
resultados['resumen']


### Paso 7 · Identificar las estaciones con más alternativas


La tabla prioriza estaciones por servicios distintos y conserva el número de paradas.

> ⚠️ Más paradas físicas no implica necesariamente más servicios diferentes.


In [ ]:
resultados['mejores_correspondencias']


### Paso 8 · Medir la sensibilidad a la fuente de coordenadas


La misma pregunta se recalcula con GTFS y bip! para cuantificar dependencia.

> ⚠️ Con bip!, tres estaciones parecerían no tener bus y 48 cambian su conteo de servicios.


In [ ]:
resultados['sensibilidad_a_la_fuente']


### Paso 9 · Separar hallazgo de explicación


El cierre publica evidencia y límites sin atribuir causas no observadas.

> ⚠️ Un desvío demuestra inconsistencia entre fuentes, no cuál de las dos es correcta ni por qué.


In [ ]:
resultados['detector_4_contra_gtfs']


## 4 · El resultado

Las 126 estaciones GTFS tienen al menos una parada de bus a 200 m y la mediana es 13 servicios. Pero 48 estaciones cambian su conteo al usar la coordenada bip!: la fuente es parte del método.

## 5 · Qué queda aprendido

1. Dos conteos distintos pueden ser correctos si sus filas tienen granos diferentes.
2. El desacuerdo entre criterios es información para revisar, no ruido que deba borrarse.
3. Toda prueba de calidad tiene un punto ciego medible.
4. Una inconsistencia entre fuentes no autoriza a declarar cuál es la verdadera.
